# Radiometric emitter (RET) GUI example

Click the RET well in a **popup window**, then this notebook analyzes it.
Plots and images stay **inside the notebook**.

**How to run**

1. Run the cells from top to bottom.
2. In *Choose your image*, either keep the example image or uncomment `IMAGE_PATH`.
3. Choose a matching `DARK_FRAME_PATH`, or set it to `None` for light-frame-only input.
4. When the well-selection window opens, click the RET well, then press **Enter**.
5. The remaining cells show plots here. If saving is on, files go in a results folder.


## 1. Setup

This cell loads QAL and keeps later plots inside the notebook.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qal import RetAnalysisConfig, RetAnalyzer, WellDetector, WellPlotter, select_wells_gui
from qal.data import ret_sample_1_df, ret_sample_1_lf, ret_sample_2_df, ret_sample_2_lf
from qal.rta.roi_extraction.well_detector_gui import use_inline_plots, use_native_window_backend
from qal.util import describe_stored_values, load_grayscale_image

from qal.examples._output import save_sample_sources, save_target_metadata


## 2. Choose your image

`IMAGE_PATH = ret_sample_1_lf()` loads the built-in example light frame.

To use your own image, comment out that line and set `IMAGE_PATH` to a
TIFF, BMP, PNG, or JPEG file path. Leave only one image option active.
The relative file path works from this notebook's folder in a GitHub checkout or download.

`IMAGE_PATH = None` raises an error because no light frame has been supplied.

**Optional dark frame:** `DARK_FRAME_PATH = ret_sample_1_df()` loads the dark
frame matching sample 1. To use a light frame only, set `DARK_FRAME_PATH = None`.
This reports raw mean intensity and standard deviation without baseline or CNR.
For your own light frame, supply its matching measured dark frame or use `None`.
Leave only one dark-frame option active.

For sample 2, use `ret_sample_2_lf()` and, if baselining, `ret_sample_2_df()`.
Matching light and dark frames must have the same dimensions and acquisition
settings. The signal ROI is also measured in the dark frame; there is no
separate dark-frame click.

**Quantitative image input:** this example passes the input through
`load_grayscale_image` before analysis. Native 2-D grayscale values and dtype
are preserved without rescaling. RGB/RGBA inputs use scikit-image rgb2gray luminance weights; alpha is ignored.


In [ ]:
# Images are loaded and converted to grayscale with load_grayscale_image.
# Already-grayscale images remain unchanged: stored values and dtype are preserved.
# RGB/RGBA inputs use scikit-image rgb2gray luminance weights; alpha is ignored.
# Choose one light-frame option; leave the others commented out.
# Uncomment the next line and put the path to your image:
# IMAGE_PATH = "/path/to/your/light_frame.tiff"

# Example using a file that ships with QAL:
# Run from this example's directory in a GitHub checkout or download.
# IMAGE_PATH = "../../data/RET_images/ret_sample_1_lf.tiff"

# Selecting the built-in example image:
IMAGE_PATH = ret_sample_1_lf()
# IMAGE_PATH = ret_sample_2_lf()

# Choose one dark-frame option; leave the others commented out.
# For light-frame-only input, comment out the active dark frame and use None:
# DARK_FRAME_PATH = None

# Uncomment the next line and put the path to your matching dark frame:
# DARK_FRAME_PATH = "/path/to/your/dark_frame.tiff"

# Example using a file that ships with QAL:
# Run from this example's directory in a GitHub checkout or download.
# DARK_FRAME_PATH = "../../data/RET_images/ret_sample_1_df.tiff"

# Selecting the dark frame that matches the built-in example image:
DARK_FRAME_PATH = ret_sample_1_df()
# DARK_FRAME_PATH = ret_sample_2_df()


## 3. Saving plots and data

These are the knobs that control whether files are written, and what they are
called. When `SAVE_RESULTS` is `True`:

- If you set `IMAGE_PATH` to a file path, files go next to that image in `<image_name>_results/`
- If you use the built-in image, files go in `ret_sample_1_lf_results/` under the notebook's working directory

Set `SAVE_SOURCE_IMAGES = True` to also copy the original built-in sample
files. User-supplied images are not copied. Multiple source images are grouped
in `source_images/`. This does not move or change cached data.


In [ ]:
# Set this to True to write plots and CSV files.
SAVE_RESULTS = True
# Optional: copy original built-in samples; user-supplied images are not copied.
SAVE_SOURCE_IMAGES = False

# File names inside the results folder (used only when SAVE_RESULTS is True):
SAVE_ROI_PLOT = "roi.png"
SAVE_STATS_CSV = "stats.csv"


## 4. Analysis options

You can usually leave these as they are.

Click the single RET well in the light frame. Automatic refinement finds the
circular well near your click. Manual coordinates use the exact click and the
diameter below; a single RET well has no neighboring wells from which to
estimate its diameter.


In [ ]:
# How wells are selected. Uncomment one:
MODE = "automatic_refinement"
# MODE = "manual_coordinates"

# Circle size in pixels. Required for manual_coordinates.
MANUAL_ROI_DIAMETER = 250.0

# Search distance in pixels around the click in automatic_refinement mode.
SEARCH_RADIUS = 300.0

# Fraction of the well used for intensity (0.5 is recommended).
REGION_OF_WELL_TO_ANALYZE = 0.5

# CNR threshold used only when a matching dark frame is supplied.
CNR_THRESHOLD = 3.0


## 5. Load the image


In [ ]:
if IMAGE_PATH is None:
    raise ValueError("No image has been supplied. Set IMAGE_PATH to an image file path or a built-in example call.")
if MODE not in {"automatic_refinement", "manual_coordinates"}:
    raise ValueError("Choose automatic_refinement or manual_coordinates.")

image = load_grayscale_image(IMAGE_PATH)
print(f"Signal: {describe_stored_values(image).summary()}")
# A light frame alone is valid; skip dark-frame loading when None is selected.
dark_frame = None
if DARK_FRAME_PATH is not None:
    dark_frame = load_grayscale_image(DARK_FRAME_PATH)
if dark_frame is not None:
    if dark_frame.shape != image.shape:
        raise ValueError("The light frame and dark frame must have the same dimensions.")
    print(f"Dark: {describe_stored_values(dark_frame).summary()}")

if isinstance(IMAGE_PATH, np.ndarray):
    results_dir = Path("ret_sample_1_lf_results")
else:
    image_file = Path(IMAGE_PATH).expanduser()
    results_dir = image_file.parent / f"{image_file.stem}_results"
if SAVE_RESULTS:
    print(f"Results folder: {results_dir.resolve()}")
else:
    results_dir = None

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(image, cmap="gray")
ax.set_title("Loaded image")
ax.axis("off")
plt.show()


## 6. Select the RET well (popup window)

This cell opens a window. Click the RET well, then press **Enter**. Drag a `+` to move it.

Later plots stay in this notebook.

Right-click near the marker to remove it. Closing an unfinished selection
cancels it. The popup needs a local graphical desktop.


In [ ]:
detector = WellDetector(parallel_processing=False)
wells = None
try:
    use_native_window_backend()
    wells = select_wells_gui(
        image,
        expected_count=1,
        well_ids=["RET"],
        detector=detector,
        search_radius=SEARCH_RADIUS,
        coordinate_mode="detect" if MODE == "automatic_refinement" else "manual",
        manual_roi_diameter=MANUAL_ROI_DIAMETER if MODE == "manual_coordinates" else None,
        enable_live_grid_preview=True,
    )
finally:
    use_inline_plots()


## 7. Analyze the selected well


In [ ]:
if not isinstance(wells, pd.DataFrame) or wells.empty:
    raise RuntimeError("No RET well selected. Rerun the selection cell and confirm with Enter.")

analyzer = RetAnalyzer(
    analysis_config=RetAnalysisConfig(
        region_of_well_to_analyze=REGION_OF_WELL_TO_ANALYZE,
        cnr_threshold=CNR_THRESHOLD,
    )
)
result = analyzer.analyze(image, wells, dark_frame=dark_frame)
stats = result.stats_df
if not result.is_baselined:
    print("No dark frame supplied; baseline and CNR were not calculated.")
stats


## 8. Show the region of interest


In [ ]:
if results_dir is not None:
    results_dir.mkdir(parents=True, exist_ok=True)

plotter = WellPlotter(stats, image=image)
roi_path = results_dir / SAVE_ROI_PLOT if results_dir is not None else None
plotter.visualize_roi(save_plot=roi_path)
if results_dir is not None:
    print(f"Saved {roi_path}")


## 9. Save the analysis table as CSV

This writes the post-analysis metrics (including well coordinates) into the
results folder, the same way `ret_gui_example.py` does. Coordinates are
already included in `stats.csv`; no duplicate coordinate table is written.

Light-frame-only results contain raw intensity and standard deviation.
Dark-frame baseline and CNR columns are included only when a dark frame is supplied.


In [ ]:
if results_dir is None:
    print("SAVE_RESULTS is False, so no CSV files were written.")
else:
    stats_path = results_dir / SAVE_STATS_CSV
    stats.to_csv(stats_path, index=False)
    print(f"Saved {stats_path}")

    if SAVE_SOURCE_IMAGES:
        save_sample_sources(results_dir, IMAGE_PATH, DARK_FRAME_PATH)
